In [1]:
# =========================================================
# 1. Imports
# =========================================================
!nvidia-smi
import csv
import re
import openpyxl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import spacy
import gensim
import gensim.corpora as corpora
import transformers

from torch import bfloat16
from huggingface_hub import login
from huggingface_hub import create_repo
from huggingface_hub import HfApi
from huggingface_hub import snapshot_download
import os
import pickle
import json

from transformers import pipeline
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_similarity

from bertopic import BERTopic
from bertopic.representation import TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer

from sentence_transformers import SentenceTransformer
from umap import UMAP
from gensim.models.coherencemodel import CoherenceModel

from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer


# =========================================================
# 2. NLTK Resources
# =========================================================
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")


Sun Mar 15 10:09:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:1E.0 Off |                    0 |
| N/A   27C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[nltk_data] Downloading package wordnet to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
from huggingface_hub import login

with open("../config/hf_token.txt") as f:
    hf_token = f.read().strip()

login(token=hf_token)

In [3]:
# =========================================================
# 3. Basic Config
# =========================================================
model_id = "meta-llama/Llama-2-7b-chat-hf"

input_excel_path = "../data/preprocessed_document.xlsx"
custom_stop_words_path = "../data/custom_stop_words.xlsx"
temp_csv_path = "sample.csv"

preprocessed_output_path = "preprocessed_test.xlsx"
embedding_output_path = "sentence_embeddings_with_text.xlsx"
topics_over_time_output_path = "topics_over_time.xlsx"

# 這裡改成你的「申請日期」欄位名稱
# 例如可能是 "Application Date"、"Priority Date"、"Patent Date"
publication_date_col = "Publication Date"


# =========================================================
# 4. Read Excel -> Convert to CSV -> Read CSV
#    (保留你原本流程)
# =========================================================
mydata = pd.read_excel(input_excel_path)

csv_writer = csv.writer(open(temp_csv_path, "w", newline="", encoding="utf-8-sig"))
workbook_active_sheet = openpyxl.load_workbook(input_excel_path).active

for row_cells in workbook_active_sheet.rows:
    row_values = [cell.value for cell in row_cells]
    csv_writer.writerow(row_values)

data = pd.read_csv(temp_csv_path)

test = data["Processed"].fillna("")


# =========================================================
# 5. Build Time Variable for Dynamic Topic Modeling
#    時間 = 專利申請年
# =========================================================
# 若欄位是完整日期，轉 datetime 再取 year
# 若欄位本身就是年份，to_datetime 也通常可處理
publication_date_raw = data[publication_date_col]

publication_date = pd.to_datetime(publication_date_raw, errors="coerce")
publication_year = publication_date.dt.year

# 移除沒有文件內容或沒有年份的資料
valid_mask = test.notna() & publication_year.notna()

test = test[valid_mask].reset_index(drop=True)
publication_year = publication_year[valid_mask].astype(int).reset_index(drop=True)

# BERTopic topics_over_time 的 timestamps 要和 documents 一一對應
timestamps = publication_year.tolist()

print("文件數量:", len(test))
print("年份範圍:", publication_year.min(), "to", publication_year.max())

文件數量: 4117
年份範圍: 2003 to 2023


In [4]:
# =========================================================
# 6. spaCy + Stopwords
# =========================================================
nlp = spacy.load("en_core_web_sm")

spacy_stopwords = spacy.lang.en.stop_words.STOP_WORDS
custom_stop_words_df = pd.read_excel(custom_stop_words_path, header=None)

excel_stop_words = custom_stop_words_df[0].tolist()
excel_stop_words = [str(word).strip().lower() for word in excel_stop_words if pd.notna(word)]

spacy_stopwords.update(excel_stop_words)

print(f"Total stop words: {len(spacy_stopwords)}")

for word in spacy_stopwords:
    if isinstance(word, str):
        nlp.vocab[word].is_stop = True

print("Custom stop words have been added successfully.")

stop_words_list = list(spacy_stopwords)


# =========================================================
# 7. Lemmatizer + POS Mapping
# =========================================================
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(pos_tag):
    """Map spaCy POS tag to WordNet POS tag."""
    tag_dict = {
        "ADJ": wordnet.ADJ,
        "NOUN": wordnet.NOUN,
        "VERB": wordnet.VERB,
        "ADV": wordnet.ADV,
    }
    return tag_dict.get(pos_tag, wordnet.NOUN)


# =========================================================
# 8. Text Preprocessing
# =========================================================
def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = re.sub(r"\s+", " ", re.sub(r"\d", "", re.sub(r"[^\w\s]", "", text.lower())))
    doc = nlp(text)

    lemmatized_tokens = [
        lemmatizer.lemmatize(token.text, get_wordnet_pos(token.pos_))
        for token in doc
        if token.text not in stop_words_list
    ]

    return " ".join(lemmatized_tokens)

preprocessed_test = [preprocess_text(doc) for doc in test]

df_preprocessed = pd.DataFrame({
    "Publication_Year": publication_year,
    "Preprocessed_test": preprocessed_test
})
df_preprocessed.to_excel("../outputs/preprocessed_test.xlsx", index=False)

print(f"預處理後的文檔已保存到 {preprocessed_output_path}")

Total stop words: 27182
Custom stop words have been added successfully.
預處理後的文檔已保存到 preprocessed_test.xlsx


In [5]:
# =========================================================
# 9. BitsAndBytes 4-bit Quantization Config
# =========================================================
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16,
)


# =========================================================
# 10. Hugging Face Login + Load Llama 2
# =========================================================
"""
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
"""

tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

generator = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=600,
    repetition_penalty=1.1,
)


# =========================================================
# 11. Quick Test Generation
# =========================================================
prompt = "Could you explain to me how 4-bit quantization works as if I am 5?"
res = generator(prompt)
print(res[0]["generated_text"])

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=600) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Could you explain to me how 4-bit quantization works as if I am 5?
 everybody knows that 4 is a small number, right? so when we take a big number and divide it by 4, we get a smaller number. like if i have 10 cookies and i want to share them equally among 4 of my friends, i will give each friend 2.5 cookies. but what happens when we take a really big number and divide it by 4? like if i had 1000 cookies and wanted to share them with 4 of my friends? well, if i divided them equally among 4 friends, each friend would get 250 cookies! that's a lot of cookies! but if i took a really big number like 1000 and divided it by 4, i would get a much smaller number called 250. so 4-bit quantization is like taking a really big number and dividing it by 4 to make it smaller. but the thing is, sometimes the number we get after dividing by 4 might not be a whole number. like if i had 1000 cookies and divided them by 4, i would get 250. but if i wanted to give each friend an exact number of cookies, li

In [6]:
system_prompt_label = """
<s>[INST] <<SYS>>
You are an expert assistant for analyzing patent technologies.
Your task is to read patent documents that belong to the same topic cluster
and generate a short topic label.

The label should:
- capture the shared technological theme
- be concise and specific
- contain no more than 8 words
- avoid punctuation-heavy or sentence-style outputs

Return only the label text.
<</SYS>>
"""

example_prompt_label = """
I have a topic that contains the following patent documents:

- A flow control valve system for fuel cell applications includes a rotating valve structure and shaft mechanism designed to improve sealing performance while minimizing abrasion between the valve seat and valve structure.
- A cathode oxygen deficiency control method for fuel cell systems adjusts controller power consumption and heater operation to improve durability and prevent performance deterioration.
- A fuel cell automobile power system integrates an inverter, motor, rechargeable energy source, and cable configuration to improve reliability.

The topic is described by the following keywords:
'fuel cell, valve system, power control, durability, inverter, sealing, vehicle'

Based on the information above, give a short label for this topic.
[/INST]
Fuel Cell Power Control Systems
"""

main_prompt_label = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords:
'[KEYWORDS]'.

Based on the information above, give a short label for this topic.
[/INST]
"""

prompt_label = system_prompt_label + example_prompt_label + main_prompt_label

In [7]:
# =========================================================
# 12. Prompt Design for BERTopic Topic Representation
# =========================================================
system_prompt = """
<s>[INST] <<SYS>>
You are an expert assistant for analyzing patent technologies.
Your task is to read multiple patent documents that belong to the same topic cluster and generate a concise technical summary describing the common technological theme.

Focus on:
- the shared engineering concepts
- the main technological innovation
- the primary application domain

Avoid describing documents individually. Instead, synthesize the overall technology represented by the topic.

Write the summary in clear and concise technical language.
<</SYS>>
"""

example_prompt = """
<s>[INST]
I have a topic that contains the following patent documents:
- A flow control valve system for fuel cell applications includes a rotating valve structure and shaft mechanism designed to improve sealing performance while minimizing abrasion between the valve seat and valve structure. A lip seal is used to enhance sealing reliability and suppress sliding wear during valve operation.
- A fuel cell automobile power system integrates an inverter, motor, rechargeable energy source, and cable configuration. Independent power cables connect the inverter and power source to improve reliability, simplify maintenance, and reduce phase differences in the electrical distribution system.
- A cathode oxygen deficiency control method for fuel cell systems adjusts controller power consumption and heater operation to improve durability and prevent performance deterioration, particularly during cold-start and regenerative braking conditions.
- A regenerative peroxide-based energy storage system generates electricity through catalytic reactions using peroxide solutions and water. The system integrates hydrogen tanks, oxygen tanks, and catalytic energy conversion units to provide efficient electrical power generation.
- A closed-system reactant and waste storage apparatus maintains balance between reactant storage and waste storage using pressure-resistant materials and composite structures. The system supports underwater or autonomous energy generation devices using hydrogen-generating reactions.

The topic is described by the following keywords:
'fuel cell, hydrogen energy, valve system, power control, energy storage, reactor system, catalytic reaction, hydrogen generation, battery technology'

Please analyze the documents and keywords above and summarize the **technological theme** of this topic.

Your summary should:
- focus on the core engineering concepts shared across these patents
- describe the main technological innovation
- explain the primary application domain

Write the summary in concise sentences.
[/INST]
This topic focuses on engineering technologies related to hydrogen-based energy systems and fuel cell power management. The inventions involve control valves, catalytic reactors, hydrogen generation mechanisms, and advanced energy storage systems such as redox flow batteries and chemical energy cycles designed to improve energy conversion efficiency, durability, and system reliability. These technologies are primarily applied in hydrogen fuel cell vehicles, autonomous power systems, and renewable energy storage applications.
"""

main_prompt = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords:
'[KEYWORDS]'.
Based on the information above, summarize the technological theme of this topic.
[/INST]
"""

prompt_summary = system_prompt + example_prompt + main_prompt

llama2 = TextGeneration(generator, prompt=prompt)


In [8]:
llama2_label = TextGeneration(
    generator,
    prompt=prompt_label,
    nr_docs=5,
    diversity=0.2
)

llama2_summary = TextGeneration(
    generator,
    prompt=prompt_summary,
    nr_docs=5,
    diversity=0.2
)

In [9]:
# =========================================================
# 13. Sentence Embedding
# =========================================================
# sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
sentence_model = SentenceTransformer("all-mpnet-base-v2")

embeddings = sentence_model.encode(preprocessed_test, show_progress_bar=True)


# =========================================================
# 14. UMAP Dimensionality Reduction
# =========================================================
umap_model = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

umap_embeddings = umap_model.fit_transform(embeddings)


# =========================================================
# 15. Save Embeddings
# =========================================================
df_embeddings = pd.DataFrame(embeddings)
df_embeddings.insert(0, "Text", preprocessed_test)
df_embeddings.to_excel(embedding_output_path, index=False)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/129 [00:00<?, ?it/s]

In [10]:
# =========================================================
# 16. Topic Diversity Function
# =========================================================
def calculate_topic_diversity(top_words_per_topic):
    all_words = [word for topic in top_words_per_topic for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words)
    return diversity

In [11]:
# =========================================================
# 17. Topic Modeling Evaluation
# =========================================================
coherence_scores = {}
diversity_scores = {}

for num_topics in [18]:
    cluster_model = KMeans(n_clusters=num_topics, random_state=42)

    vectorizer_model = CountVectorizer(
        stop_words=stop_words_list,
        ngram_range=(1, 2),
        lowercase=True,
        min_df=10
    )

    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

    representation_model = {
        "Summary": llama2_summary
    }

    topic_model = BERTopic(
        embedding_model=sentence_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        top_n_words=15,
        verbose=True,
        calculate_probabilities=False,
    )

    topics, probs = topic_model.fit_transform(preprocessed_test, embeddings)

    # ---- Coherence (Embedding-based semantic coherence) ----
        # ---- Coherence (Embedding-based semantic coherence) ----
    topic_data = topic_model.get_topics()
    topic_words = []

    for topic_id, topic_terms in topic_data.items():
        if topic_id == -1:
            continue
        if topic_terms is None or topic_terms is False:
            continue
        if not isinstance(topic_terms, list) or len(topic_terms) == 0:
            continue

        words = [word for word, _ in topic_terms[:15] if isinstance(word, str) and word.strip() != ""]
        if len(words) >= 2:
            topic_words.append(words)

    if len(topic_words) == 0:
        coherence = None
    else:
        topic_scores = []

        for words in topic_words:
            word_embeddings = sentence_model.encode(words, show_progress_bar=False)
            centroid = np.mean(word_embeddings, axis=0, keepdims=True)
            sims = cosine_similarity(word_embeddings, centroid).flatten()
            topic_scores.append(float(np.mean(sims)))

        coherence = float(np.mean(topic_scores)) if len(topic_scores) > 0 else None
    coherence_scores[num_topics] = coherence

    # ---- Diversity ----
    topic_data = topic_model.get_topics()
    top_n = 15
    top_words_per_topic = []

    for topic_id, topic_terms in topic_data.items():
        if topic_id == -1:
            continue
        if topic_terms is None or topic_terms is False:
            continue
        if not isinstance(topic_terms, list) or len(topic_terms) == 0:
            continue

        words = [word for word, _ in topic_terms[:top_n] if isinstance(word, str) and word.strip() != ""]
        if len(words) > 0:
            top_words_per_topic.append(words)

    diversity = calculate_topic_diversity(top_words_per_topic) if len(top_words_per_topic) > 0 else None
    diversity_scores[num_topics] = diversity

    print(
        f"Number of Topics: {num_topics}, "
        f"Coherence Score: {coherence}, "
        f"Diversity Score: {diversity}"
    )

print(coherence_scores)
print(diversity_scores)

2026-03-15 10:17:21,020 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-15 10:17:28,546 - BERTopic - Dimensionality - Completed ✓
2026-03-15 10:17:28,547 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-15 10:17:28,672 - BERTopic - Cluster - Completed ✓
2026-03-15 10:17:28,684 - BERTopic - Representation - Fine-tuning topics using representation models.
 50%|█████     | 9/18 [04:43<04:57, 33.01s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=600) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 18/18 [09:35<00:00, 31.95s/it]
2026-03-15 10:27:04,225 - BERTopic - Representation - Completed ✓


Number of Topics: 18, Coherence Score: 0.5577623479896121, Diversity Score: 0.7148148148148148
{18: 0.5577623479896121}
{18: 0.7148148148148148}


In [12]:
# =========================================================
# 18. Save Evaluation Results to Excel
# =========================================================
results_df = pd.DataFrame({
    "Number_of_Topics": list(coherence_scores.keys()),
    "Coherence_Score": list(coherence_scores.values()),
    "Diversity_Score": [diversity_scores[k] for k in coherence_scores.keys()]
})

# 依 topic 數排序
results_df = results_df.sort_values("Number_of_Topics").reset_index(drop=True)

# 存成 Excel
results_df.to_excel("topic_model_evaluation.xlsx", index=False)

print(results_df)
print("已儲存為 topic_model_evaluation.xlsx")


'\n# =========================================================\n# 18. Save Evaluation Results to Excel\n# =========================================================\nresults_df = pd.DataFrame({\n    "Number_of_Topics": list(coherence_scores.keys()),\n    "Coherence_Score": list(coherence_scores.values()),\n    "Diversity_Score": [diversity_scores[k] for k in coherence_scores.keys()]\n})\n\n# 依 topic 數排序\nresults_df = results_df.sort_values("Number_of_Topics").reset_index(drop=True)\n\n# 存成 Excel\nresults_df.to_excel("topic_model_evaluation.xlsx", index=False)\n\nprint(results_df)\nprint("已儲存為 topic_model_evaluation.xlsx")\n'

In [13]:
topics_over_time = topic_model.topics_over_time(
        docs=preprocessed_test,
        timestamps=timestamps,
        nr_bins=len(sorted(set(timestamps)))
    )

21it [00:00, 40.96it/s]


In [14]:
# =========================================================
# 18. Save BERTopic Model + Reproducible Artifacts
# =========================================================

save_dir = "bertopic_model_18"
os.makedirs(save_dir, exist_ok=True)

# -------------------------
# 1. Save BERTopic model as pkl
# -------------------------
model_pkl_path = os.path.join(save_dir, "bertopic_model.pkl")
with open(model_pkl_path, "wb") as f:
    pickle.dump(topic_model, f)

print(f"✅ Saved model: {model_pkl_path}")

# -------------------------
# 2. Save embeddings as .npy
# -------------------------
embeddings_npy_path = os.path.join(save_dir, "embeddings.npy")
np.save(embeddings_npy_path, embeddings)
print(f"✅ Saved embeddings: {embeddings_npy_path}")

# -------------------------
# 3. Prepare document_embeddings sheet
# -------------------------
df_document_embeddings = pd.DataFrame(embeddings)
df_document_embeddings.insert(0, "Document", preprocessed_test)
df_document_embeddings.insert(1, "Topic", topics)
df_document_embeddings.insert(2, "Timestamp", timestamps)

embedding_cols = ["Document", "Topic", "Timestamp"] + [f"emb_{i}" for i in range(embeddings.shape[1])]
df_document_embeddings.columns = embedding_cols

# -------------------------
# 4. Prepare topics_over_time sheet
# -------------------------
df_topics_over_time = topics_over_time.copy()

# -------------------------
# 5. Save Excel with sheets
# -------------------------
excel_output_path = os.path.join(save_dir, "bertopic_outputs.xlsx")
with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    df_document_embeddings.to_excel(writer, sheet_name="document_embeddings", index=False)
    df_topics_over_time.to_excel(writer, sheet_name="topics_over_time", index=False)

print(f"✅ Saved Excel with sheets: {excel_output_path}")

# -------------------------
# 6. Save topic_info as xlsx + csv
# -------------------------
df_topic_info = topic_model.get_topic_info()

topic_info_xlsx_path = os.path.join(save_dir, "topic_info.xlsx")
df_topic_info.to_excel(topic_info_xlsx_path, index=False)

topic_info_csv_path = os.path.join(save_dir, "topic_info.csv")
df_topic_info.to_csv(topic_info_csv_path, index=False, encoding="utf-8-sig")

print(f"✅ Saved topic info xlsx: {topic_info_xlsx_path}")
print(f"✅ Saved topic info csv : {topic_info_csv_path}")

# -------------------------
# 7. Save topic labels and summaries
# -------------------------
# BERTopic representation_model 裡有 "Main" 和 "Summary"
# get_topic() 回傳格式可能依 representation 不同而有差異
# 這裡用 topic labels / custom labels / topic info 盡量穩定保存

topic_labels_records = []
topic_summary_records = []

# 先取 topic info 方便合併 Count 等資訊
topic_info_lookup = {}
for _, row in df_topic_info.iterrows():
    topic_info_lookup[row["Topic"]] = row.to_dict()

# 取得所有 topic id
all_topic_ids = sorted([topic_id for topic_id in topic_model.get_topics().keys() if topic_id != -1])

for topic_id in all_topic_ids:
    # keywords
    topic_terms = topic_model.get_topic(topic_id)
    keywords = ", ".join([word for word, _ in topic_terms[:15]]) if topic_terms else ""

    # 預設值
    main_label = None
    summary_text = None

    # 嘗試從 topic aspect representations 取值
    try:
        aspect_repr = topic_model.topic_aspects_
        if "Main" in aspect_repr and topic_id in aspect_repr["Main"]:
            main_items = aspect_repr["Main"][topic_id]
            if isinstance(main_items, list) and len(main_items) > 0:
                # TextGeneration 常見輸出: [('label text', score)] 或 ['label text']
                first_item = main_items[0]
                if isinstance(first_item, tuple):
                    main_label = first_item[0]
                else:
                    main_label = str(first_item)

        if "Summary" in aspect_repr and topic_id in aspect_repr["Summary"]:
            summary_items = aspect_repr["Summary"][topic_id]
            if isinstance(summary_items, list) and len(summary_items) > 0:
                first_item = summary_items[0]
                if isinstance(first_item, tuple):
                    summary_text = first_item[0]
                else:
                    summary_text = str(first_item)
    except Exception:
        pass

    # 若 Main label 抓不到，退回 topic label
    if main_label is None:
        try:
            # 例如 "18_fuel_cell_power_control"
            main_label = topic_model.topic_labels_[topic_id]
        except Exception:
            main_label = None

    # 若 summary 還抓不到，先留空
    if summary_text is None:
        summary_text = ""

    # count
    topic_count = topic_info_lookup.get(topic_id, {}).get("Count", None)
    topic_name = topic_info_lookup.get(topic_id, {}).get("Name", None)

    topic_labels_records.append({
        "Topic": topic_id,
        "Count": topic_count,
        "Name": topic_name,
        "Label": main_label,
        "Keywords": keywords
    })

    topic_summary_records.append({
        "Topic": topic_id,
        "Count": topic_count,
        "Name": topic_name,
        "Summary": summary_text,
        "Keywords": keywords
    })

df_topic_labels = pd.DataFrame(topic_labels_records)
df_topic_summaries = pd.DataFrame(topic_summary_records)

topic_labels_csv_path = os.path.join(save_dir, "topic_labels.csv")
topic_summary_csv_path = os.path.join(save_dir, "topic_summary.csv")

df_topic_labels.to_csv(topic_labels_csv_path, index=False, encoding="utf-8-sig")
df_topic_summaries.to_csv(topic_summary_csv_path, index=False, encoding="utf-8-sig")

print(f"✅ Saved topic labels : {topic_labels_csv_path}")
print(f"✅ Saved topic summary: {topic_summary_csv_path}")


'\n# =========================================================\n# 18. Save BERTopic Model + Reproducible Artifacts\n# =========================================================\n\nsave_dir = "bertopic_model_18"\nos.makedirs(save_dir, exist_ok=True)\n\n# -------------------------\n# 1. Save BERTopic model as pkl\n# -------------------------\nmodel_pkl_path = os.path.join(save_dir, "bertopic_model.pkl")\nwith open(model_pkl_path, "wb") as f:\n    pickle.dump(topic_model, f)\n\nprint(f"✅ Saved model: {model_pkl_path}")\n\n# -------------------------\n# 2. Save embeddings as .npy\n# -------------------------\nembeddings_npy_path = os.path.join(save_dir, "embeddings.npy")\nnp.save(embeddings_npy_path, embeddings)\nprint(f"✅ Saved embeddings: {embeddings_npy_path}")\n\n# -------------------------\n# 3. Prepare document_embeddings sheet\n# -------------------------\ndf_document_embeddings = pd.DataFrame(embeddings)\ndf_document_embeddings.insert(0, "Document", preprocessed_test)\ndf_document_e

In [15]:
df_documents = pd.DataFrame({
    "Document": list(test),
    "Preprocessed_Document": list(preprocessed_test),
    "Topic": list(topics),
    "Timestamp": list(timestamps)
})
df_documents.to_csv(os.path.join(save_dir, "documents.csv"), index=False, encoding="utf-8-sig")'''

'df_documents = pd.DataFrame({\n    "Document": list(test),\n    "Preprocessed_Document": list(preprocessed_test),\n    "Topic": list(topics),\n    "Timestamp": list(timestamps)\n})\ndf_documents.to_csv(os.path.join(save_dir, "documents.csv"), index=False, encoding="utf-8-sig")'

In [16]:
with open("../config/hf_token.txt") as f:
    hf_token = f.read().strip()

login(token=hf_token)

In [17]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '67c7c84ba0754aaa5118cf33', 'name': 'jainxin', 'fullname': 'You', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/uIbHsgfuP8wluoXsDMgpu.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'upload_model', 'role': 'fineGrained', 'createdAt': '2026-03-15T09:01:31.346Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '67c7c84ba0754aaa5118cf33', 'type': 'user', 'name': 'jainxin'}, 'permissions': ['repo.access.read', 'repo.content.read', 'repo.write', 'collection.read', 'collection.write']}]}}}}


In [18]:
from huggingface_hub import create_repo

repo_id = "jainxin/bertopic-patent-model"

create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

print("Repo ready:", repo_id)'''

'from huggingface_hub import create_repo\n\nrepo_id = "jainxin/bertopic-patent-model"\n\ncreate_repo(\n    repo_id=repo_id,\n    repo_type="model",\n    exist_ok=True\n)\n\nprint("Repo ready:", repo_id)'

In [20]:
readme_text = """---
license: mit
tags:
- bertopic
- topic-modeling
- patent-analysis
- sentence-transformers
- llama2
language:
- en
---

# BERTopic Patent Topic Model

This repository contains a BERTopic model trained on patent documents.

## Files

- bertopic_model.pkl → saved BERTopic model
- embeddings.npy → document embeddings
- topic_info.csv → topic statistics
- topic_labels.csv → topic labels generated by Llama2
- topic_summary.csv → topic summaries generated by Llama2
- bertopic_outputs.xlsx → document embeddings + dynamic topics

## Notes

To reload the model:

```python
import pickle

with open("bertopic_model.pkl", "rb") as f:
    topic_model = pickle.load(f)
"""

SyntaxError: incomplete input (3979009547.py, line 1)

In [50]:
# 4️⃣ 上傳整個資料夾
使用 `upload_folder`：

```python
'''
'''from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="bertopic_model_18",
    repo_id="jainxin/bertopic-patent-model",
    repo_type="model"
)



Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete


In [51]:
# =========================================================
# 19. Extract Llama2-generated Topic Summaries Only
# =========================================================

full_topics = topic_model.get_topics(full=True)

topic_summaries = {}

for topic_id in full_topics["Summary"]:
    if topic_id == -1:
        continue

    summary_value = full_topics["Summary"][topic_id]

    if summary_value and len(summary_value) > 0:
        first_item = summary_value[0]

        if isinstance(first_item, tuple):
            topic_summaries[topic_id] = first_item[0].strip()
        else:
            topic_summaries[topic_id] = str(first_item).strip()

# =========================================================
# 20. Save Topic Summaries Only
# =========================================================
df_topic_summaries = pd.DataFrame([
    {
        "Topic": topic_id,
        "Summary": topic_summaries[topic_id]
    }
    for topic_id in sorted(topic_summaries.keys())
])

df_topic_summaries.to_excel("topic_summaries.xlsx", index=False)
print(df_topic_summaries)

# =========================================================
# 21. Visualization for Dynamic Topics
# =========================================================
fig_topics_over_time = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=18
)

fig_topics_over_time.write_html("topics_over_time.html")
print("Saved: topics_over_time.html")

    Topic                                            Summary
0       0  The technological theme of this topic revolves...
1       1  The technological theme of this topic revolves...
2       2  The technological theme of this topic revolves...
3       3  The technological theme of this topic appears ...
4       4  The technological theme of this topic revolves...
5       5  The technological theme of this topic revolves...
6       6  The technological theme of this topic revolves...
7       7  The technological theme of this topic revolves...
8       8  The technological theme of this topic revolves...
9       9  The technological theme of this topic revolves...
10     10  The technological theme of this topic revolves...
11     11  The technological theme of this topic is focus...
12     12  The technological theme of this topic revolves...
13     13  The technological theme of this topic appears ...
14     14  The technological theme of this topic revolves...
15     15  The technolog

In [21]:
output_path = "../outputs/topic_info.xlsx"

df_topic_info = topic_model.get_topic_info()
df_topic_info.to_excel(output_path, index=False)

print("Saved:", output_path)

Saved: ../outputs/topic_info.xlsx
